In [13]:
import pickle
import numpy as np
import os

directory_path = '/data1/guanren/CSL-Daily/csl-daily_pose/S006108_P0000_T00'

body_pose_list = []
left_hand_pose_list = []
right_hand_pose_list = []
transl_list = []
beta_list = []
global_orient_list = []

for filename in sorted(os.listdir(directory_path)):
    if filename.endswith('.pkl'):
        file_path = os.path.join(directory_path, filename)
        
        with open(file_path, 'rb') as file:
            data = pickle.load(file)
        
        body_pose_list.append(np.expand_dims(data['smplx_body_pose'],axis=0))
        left_hand_pose_list.append(np.expand_dims(data['smplx_lhand_pose'],axis=0))
        right_hand_pose_list.append(np.expand_dims(data['smplx_rhand_pose'],axis=0))
        transl_list.append(np.expand_dims(data['cam_trans'],axis=0))
        beta_list.append(np.expand_dims(data['smplx_shape'],axis=0))
        global_orient_list.append(np.expand_dims(data['smplx_root_pose'],axis=0))

body_pose_concat = np.concatenate(body_pose_list, axis=0)
left_hand_pose_concat = np.concatenate(left_hand_pose_list, axis=0)
right_hand_pose_concat = np.concatenate(right_hand_pose_list, axis=0)
transl_concat = np.concatenate(transl_list, axis=0)
beta_concat = np.concatenate(beta_list, axis=0)
global_orient_concat = np.concatenate(global_orient_list, axis=0)

print("Body Pose Shape:", body_pose_concat.shape)
print("Left Hand Pose Shape:", left_hand_pose_concat.shape)
print("Right Hand Pose Shape:", right_hand_pose_concat.shape)
print("Translation Shape:", transl_concat.shape)
print("Beta Shape:", beta_concat.shape)
print("Global Orientation Shape:", global_orient_concat.shape)

body_pose_concat = body_pose_concat.reshape(-1, 21, 3)
for i in range(transl_concat.shape[0]):
    transl_concat[i, :] = transl_concat[-1, :]
    global_orient_concat[i, :] = global_orient_concat[-1, :]
    body_pose_concat[i, :9, :] = body_pose_concat[-1, :9, :]
body_pose_concat = body_pose_concat.reshape(-1, 63)

full_body = 22
body_keypoint = 14
hand_keypoint = 30
full_body_hand = 52

parent_indices = [
    -1, 0, 0, 1, 2, 3, 4, 0, 7, 7, 8, 9, 10, 11
]
# key_point_indices = [0, 1, 2, 4, 5, 7, 8, 9, 16, 17, 18, 19, 20, 21, 34, 35, 36,22, 23, 24, 25, 26, 27, 31,32,33,28,29,30, 49,50,51, 37, 38, 39,40,41,42,46,47,48, 43, 44,45]
key_point_indices = [0, 1, 2, 4, 5, 7, 8, 9, 16, 17, 18, 19, 20, 21]

smplx_hand_to_panoptic = [0, 13, 14, 15, 16, 1, 2, 3, 17, 4, 5, 6, 18, 10, 11, 12, 19, 7, 8, 9, 20]

right_hand_idxs = [21] + list(range(40, 55)) + list(range(71, 76))
left_hand_idxs = [20] + list(range(25, 40)) + list(range(66, 71))
print(right_hand_pose_concat.reshape(-1, 15, 3))

Body Pose Shape: (64, 63)
Left Hand Pose Shape: (64, 45)
Right Hand Pose Shape: (64, 45)
Translation Shape: (64, 3)
Beta Shape: (64, 10)
Global Orientation Shape: (64, 3)
[[[ 0.03190429 -0.2083032   0.03414434]
  [ 0.06238078  0.09221315 -0.12758654]
  [ 0.00704646  0.02984706  0.02796525]
  ...
  [ 0.07300866 -0.05728114 -0.07624105]
  [-0.06979775  0.04441589  0.1114053 ]
  [-0.08663279  0.00923815 -0.10308369]]

 [[ 0.00361849 -0.1635354  -0.12085643]
  [-0.0318994   0.07394984 -0.35587773]
  [ 0.04641252 -0.0111991  -0.15751757]
  ...
  [-0.05909663 -0.05113706 -0.10767127]
  [-0.05319589  0.02344006  0.06558895]
  [-0.09739575  0.05192614 -0.20565668]]

 [[-0.0312768  -0.20552002 -0.03444993]
  [ 0.00688543  0.06459168 -0.3269241 ]
  [ 0.05160971 -0.0123034  -0.17978027]
  ...
  [-0.16999662 -0.01992357 -0.17523581]
  [-0.05303091  0.007651    0.12680534]
  [-0.12191898  0.04875239 -0.31123376]]

 ...

 [[ 0.01195936 -0.08059198  0.151993  ]
  [ 0.0550639   0.06220675 -0.38146526]

In [14]:
import smplx
import torch
import numpy as np
from scipy.spatial.transform import Rotation as R

kwargs = dict(gender='neutral',
        num_betas=10,
        use_face_contour=True,
        flat_hand_mean=False,
        use_pca=False,
        batch_size=1)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
smplx_model = smplx.create(
        '/home/guanren/expressive-humanoid/body_models', 'smplx', 
        **kwargs).to(device)

# betas = np.load('beta_optimized.npy')
# beta_tensor = torch.tensor(betas, dtype=torch.float32).to(device)

rotation = R.from_euler('x', 180, degrees=True)
for i in range(body_pose_concat.shape[0]):
    original_rotation = R.from_rotvec(global_orient_concat[i,:])
    rotated_rotation = rotation * original_rotation
    global_orient_concat[i,:] = rotated_rotation.as_rotvec()
global_positions = np.zeros((body_pose_concat.shape[0], full_body, 3))
left_global_hands = np.zeros((body_pose_concat.shape[0], 21, 3))
right_global_hands = np.zeros((body_pose_concat.shape[0], 21, 3))
for i in range(body_pose_concat.shape[0]):
        betas = beta_concat[i]
        betas = np.expand_dims(betas, axis=0)
        transl = transl_concat[i]
        transl = np.expand_dims(transl, axis=0)
        pose = body_pose_concat[i]
        pose = np.expand_dims(pose, axis=0)
        global_orient = global_orient_concat[i]
        global_orient = np.expand_dims(global_orient, axis=0)
        left_hand_pose = left_hand_pose_concat[i]
        left_hand_pose = np.expand_dims(left_hand_pose, axis=0)
        right_hand_pose = right_hand_pose_concat[i]
        right_hand_pose = np.expand_dims(right_hand_pose, axis=0)

        betas = torch.tensor(betas, dtype=torch.float32).to(device)
        # print(betas.shape)
        transl = torch.tensor(transl, dtype=torch.float32).to(device)
        pose = torch.tensor(pose, dtype=torch.float32).to(device)
        global_orient = torch.tensor(global_orient, dtype=torch.float32).to(device)
        left_hand_pose = torch.tensor(left_hand_pose, dtype=torch.float32).reshape(1,-1).to(device)
        right_hand_pose = torch.tensor(right_hand_pose, dtype=torch.float32).reshape(1,-1).to(device)

        output = smplx_model(betas=betas, body_pose=pose, global_orient=global_orient, transl=transl, left_hand_pose=left_hand_pose, right_hand_pose=right_hand_pose)
        global_positions_temp = output.joints.detach().cpu().numpy().squeeze()
        left_hand_joints = global_positions_temp[left_hand_idxs, :][smplx_hand_to_panoptic, :]
        left_joint_pos = left_hand_joints - global_positions_temp[20:20 + 1, :]
        origin = left_joint_pos[0:1, :]
        left_joint_pos -= origin
        left_global_hands[i] = left_joint_pos
        right_hand_joints = global_positions_temp[right_hand_idxs, :][smplx_hand_to_panoptic, :]
        right_joint_pos = right_hand_joints - global_positions_temp[21:21 + 1, :]
        origin = right_joint_pos[0:1, :]
        right_joint_pos -= origin
        right_global_hands[i] = right_joint_pos
        global_positions_temp = global_positions_temp[:22,:]
        # global_positions_temp = np.concatenate((global_positions_temp[:22,:],global_positions_temp[25:55,:]), axis=0)
        global_positions[i] = global_positions_temp

In [15]:
delta_time = 1 / 30
global_linear_velocity = np.diff(global_positions, axis=0) / delta_time
global_linear_velocity = np.concatenate([np.zeros((1, full_body, 3)), global_linear_velocity], axis=0)
global_linear_velocity = global_linear_velocity[:, key_point_indices]

global_orient_concat = np.expand_dims(global_orient_concat, axis=1)
body_pose_concat = body_pose_concat.reshape(-1, 21, 3)
all_body_pose = np.concatenate([global_orient_concat, body_pose_concat], axis=1)
rotation = R.from_rotvec(all_body_pose.reshape(-1, 3), degrees=False)
body_pose_quat = rotation.as_quat().reshape(-1, full_body, 4)
body_pose_quat = body_pose_quat[:, key_point_indices]
global_orient_concat = np.squeeze(global_orient_concat, axis=1)

from scipy.spatial.transform import Rotation as R

def calculate_global_angular_velocity(joint_orientations, delta_time):

    num_frames, num_joints, _ = joint_orientations.shape
    global_angular_velocity = np.zeros((num_frames - 1, num_joints, 3))

    for frame in range(num_frames - 1):
        for joint in range(num_joints):

            q1 = R.from_quat(joint_orientations[frame, joint])
            q2 = R.from_quat(joint_orientations[frame + 1, joint])
            relative_rotation = q2 * q1.inv()
            rotvec = relative_rotation.as_rotvec()
            angular_velocity = rotvec / delta_time
            global_angular_velocity[frame, joint] = angular_velocity

    return global_angular_velocity

global_angular_velocity = calculate_global_angular_velocity(body_pose_quat, delta_time)
global_angular_velocity = np.concatenate([np.zeros((1, body_keypoint, 3)), global_angular_velocity], axis=0)
def calculate_local_position_from_global(joint_global_pos, parent_indices):

    num_frames, num_joints, _ = joint_global_pos.shape
    joint_local_pos = np.zeros_like(joint_global_pos)

    for j in range(num_joints):
        parent_index = parent_indices[j]
        
        if parent_index == -1:
            joint_local_pos[:, j, :] = transl_concat
        else:
            joint_local_pos[:, j, :] = joint_global_pos[:, j, :] - joint_global_pos[:, parent_index, :]

    return joint_local_pos


joint_local_pos = calculate_local_position_from_global(global_positions[:,key_point_indices,:], parent_indices)

new_order = [
    'Pelvis', 'L_Hip', 'R_Hip', 'L_Knee', 
    'R_Knee', 'L_Ankle', 'R_Ankle', 
    'Spine3', 'L_Shoulder','R_Shoulder', 'L_Elbow', 'R_Elbow', 'L_Wrist', 'R_Wrist'
]

import numpy as np
from collections import OrderedDict

skeleton_motion = OrderedDict([
    ('rotation', {'arr': body_pose_quat, 'context': {'dtype': 'float32'}}),
    ('root_translation', {'arr': transl_concat, 'context': {'dtype': 'float32'}}),
    ('global_velocity', {'arr': global_linear_velocity, 'context': {'dtype': 'float32'}}),
    ('global_angular_velocity', {'arr': global_angular_velocity, 'context': {'dtype': 'float32'}}),
    ('skeleton_tree', OrderedDict([
        ('node_names', new_order),
        ('parent_indices', {'arr': np.array(parent_indices), 'context': {'dtype': 'int64'}}),
        ('local_translation', {'arr': joint_local_pos[0], 'context': {'dtype': 'float32'}})
    ])),
    ('is_local', True),
    ('fps', 30),
    ('__name__', 'SkeletonMotion')
])

static_joint = np.load('/home/guanren/expressive-humanoid/ASE/ase/poselib/data/npy/S000003_P0000_T00.npy', allow_pickle=True)
static_joint = OrderedDict(static_joint.item())
static_joint_list = [0, 1, 2, 3, 4, 5, 6, 7]

skeleton_motion['rotation']['arr'][:,static_joint_list,:] = static_joint['rotation']['arr'][0,static_joint_list,:]
skeleton_motion['root_translation']['arr'][:] = static_joint['root_translation']['arr'][0]
skeleton_motion['skeleton_tree']['local_translation']['arr'][static_joint_list,:] = static_joint['skeleton_tree']['local_translation']['arr'][static_joint_list,:]

# hand_motion = OrderedDict([
#     ('rotation', {'arr': body_pose_quat[:,14:], 'context': {'dtype': 'float32'}}),
#     ('root_translation', {'arr': transl_concat, 'context': {'dtype': 'float32'}}),
#     ('global_velocity', {'arr': global_linear_velocity[:,14:], 'context': {'dtype': 'float32'}}),
#     ('global_angular_velocity', {'arr': global_angular_velocity[:,14:], 'context': {'dtype': 'float32'}}),
#     ('skeleton_tree', OrderedDict([
#         ('node_names', new_order[14:]),
#         ('local_translation', {'arr': joint_local_pos[0][14:], 'context': {'dtype': 'float32'}})
#     ])),
#     ('is_local', True),
#     ('fps', 30),
#     ('__name__', 'SkeletonMotion')
# ])


# np.save('hand_5.npy', hand_motion)
np.save('S001906_P0000_T00.npy', skeleton_motion)




In [16]:

import numpy as np
import pickle
from pathlib import Path
from dex_retargeting.constants import RobotName, RetargetingType, HandType, get_default_config_path
from dex_retargeting.retargeting_config import RetargetingConfig
from dex_retargeting.seq_retarget import SeqRetargeting
import tqdm 

config_path = get_default_config_path(RobotName.linker, RetargetingType.vector, HandType.left)
robot_dir = '/home/guanren/expressive-humanoid/body_models/hands'
RetargetingConfig.set_default_urdf_dir(str(robot_dir))
retargeting = RetargetingConfig.load_from_file(config_path).build()

left_retarget_hand = []
right_retarget_hand = []

def normalize(v):
    """Normalize a vector."""
    return v / np.linalg.norm(v)

def calculate_rotation_matrix_to_target(source_vector, target_vector):
    """
    Calculate a rotation matrix to align the source_vector to the target_vector.
    
    Args:
        source_vector (numpy.ndarray): Current vector to rotate (3,).
        target_vector (numpy.ndarray): Target vector to align with (3,).
    
    Returns:
        numpy.ndarray: Rotation matrix (3, 3).
    """
    source_vector = normalize(source_vector)
    target_vector = normalize(target_vector)
    cross_prod = np.cross(source_vector, target_vector)
    dot_prod = np.dot(source_vector, target_vector)
    if np.linalg.norm(cross_prod) < 1e-6:  # Already aligned
        return np.eye(3)
    
    skew_symmetric = np.array([
        [0, -cross_prod[2], cross_prod[1]],
        [cross_prod[2], 0, -cross_prod[0]],
        [-cross_prod[1], cross_prod[0], 0]
    ])
    rotation_matrix = (
        np.eye(3) + skew_symmetric +
        np.dot(skew_symmetric, skew_symmetric) * ((1 - dot_prod) / (np.linalg.norm(cross_prod) ** 2))
    )
    return rotation_matrix

def align_09_to_z_axis(global_coords):
    """
    Rotate the hand's global coordinates so that the line connecting
    Joint 0 and Joint 7 is perpendicular to the XY plane (aligned with the Z-axis).
    
    Args:
        global_coords (numpy.ndarray): Shape (n_joints, 3), global coordinates of joints.
    
    Returns:
        numpy.ndarray: Adjusted global coordinates.
    """
    joint_0 = global_coords[0]
    joint_9 = global_coords[9]
    vector_09 = joint_9 - joint_0

    target_vector = np.array([0, 0, 1])  # Z-axis

    rotation_matrix = calculate_rotation_matrix_to_target(vector_09, target_vector)

    rotated_coords = (rotation_matrix @ global_coords.T).T
    return rotated_coords

def vector_angle_with_x_axis_2d(vector):
    """
    Calculate the angle between a 2D vector (projected onto XY plane) and the positive X-axis.

    Args:
        vector (numpy.ndarray): 3D vector, only the X and Y components will be used.

    Returns:
        float: Angle in radians between the vector and the positive X-axis in the XY plane.
    """
    vector_2d = vector[:2]  # Only use X and Y components
    unit_vector = vector_2d / np.linalg.norm(vector_2d)
    x_axis = np.array([1, 0])  # Positive X-axis in 2D
    cos_angle = np.dot(unit_vector, x_axis)
    return np.arccos(np.clip(cos_angle, -1.0, 1.0))

def rotate_based_on_joint_distribution(global_coords):
    
    def calculate_rotation_matrix_2d(angle):

        return np.array([
            [np.cos(angle), -np.sin(angle), 0],
            [np.sin(angle), np.cos(angle), 0],
            [0, 0, 1]
        ])

    joint_0 = global_coords[0]
    joint_12 = global_coords[11]
    vector_012 = joint_12[:2] - joint_0[:2]
    # vector_012 = np.mean([global_coords[idx][:2] for idx in [4,8,12,16,20]], axis=0) - joint_0[:2]

    angle_with_x = np.arctan2(vector_012[1], vector_012[0])

    rotation_matrix = calculate_rotation_matrix_2d(-angle_with_x)

    global_coords[:, :2] = (rotation_matrix[:2, :2] @ global_coords[:, :2].T).T

    return global_coords

for i in range(global_positions.shape[0]):
    retargeting_type = retargeting.optimizer.retargeting_type
    indices = retargeting.optimizer.target_link_human_indices
    origin_indices = indices[0, :]
    task_indices = indices[1, :]
    temp = align_09_to_z_axis(left_global_hands[i])
    temp = rotate_based_on_joint_distribution(temp)
    ref_value = temp[task_indices, :] - temp[origin_indices, :]
    left_qpos = retargeting.retarget(ref_value)
    left_retarget_hand.append(left_qpos)

retargeting.verbose()

# left_retarget_hand = np.array(left_retarget_hand)

meta_data = dict(
            config_path=config_path,
            dof=len(retargeting.optimizer.robot.dof_joint_names),
            joint_names=retargeting.optimizer.robot.dof_joint_names,
        )

# output_path = Path('/home/guanren/expressive-humanoid/dex-retargeting/example/vector_retargeting/data/left_retarget_hand.pkl')
# with output_path.open("wb") as f:
#     pickle.dump(dict(data=left_retarget_hand, meta_data=meta_data), f)

config_path = get_default_config_path(RobotName.linker, RetargetingType.vector, HandType.right)
RetargetingConfig.set_default_urdf_dir(str(robot_dir))
retargeting = RetargetingConfig.load_from_file(config_path).build()

for i in range(global_positions.shape[0]):
    retargeting_type = retargeting.optimizer.retargeting_type
    indices = retargeting.optimizer.target_link_human_indices
    origin_indices = indices[0, :]
    task_indices = indices[1, :]
    temp = align_09_to_z_axis(right_global_hands[i])
    temp = rotate_based_on_joint_distribution(temp)
    ref_value = temp[task_indices, :] - temp[origin_indices, :]
    right_qpos = retargeting.retarget(ref_value)
    right_retarget_hand.append(right_qpos)

retargeting.verbose()
print(retargeting.optimizer.robot.dof_joint_names)

meta_data = dict(
            config_path=config_path,
            dof=len(retargeting.optimizer.robot.dof_joint_names),
            joint_names=retargeting.optimizer.robot.dof_joint_names,
        )

# output_path = Path('/home/guanren/expressive-humanoid/dex-retargeting/example/vector_retargeting/data/S007172_P0000_T00_right_retarget_hand.pkl')
# with output_path.open("wb") as f:
#     pickle.dump(dict(data=right_retarget_hand, meta_data=meta_data), f)

right_retarget_hand = np.array(right_retarget_hand)
left_retarget_hand = np.array(left_retarget_hand)
retarget_hand = np.concatenate((left_retarget_hand, right_retarget_hand), axis=1)

np.save('S006108_P0000_T00_hand_full_pose.npy', retarget_hand)



Retargeting 64 times takes: 0.42359722405672073s
Last distance: 0.00151333383267956
Retargeting 64 times takes: 0.6289751566946507s
Last distance: 0.002171679272182075
['index_joint1', 'index_joint2', 'index_joint3', 'little_joint1', 'little_joint2', 'little_joint3', 'middle_joint1', 'middle_joint2', 'middle_joint3', 'ring_joint1', 'ring_joint2', 'ring_joint3', 'thumb_joint1', 'thumb_joint3', 'thumb_joint4']


In [47]:
import numpy as np


_MANO_JOINT_CONNECT = [
    [0, 1], [1, 2], [2, 3], [3, 4], 
    [0, 5], [5, 6], [6, 7], [7, 8],
    [0, 9], [9, 10], [10, 11], [11, 12],
    [0, 13], [13, 14], [14, 15], [15, 16],
    [0, 17], [17, 18], [18, 19], [19, 20]
]

import plotly.graph_objects as go
right_global_hands_vis = right_global_hands[50]
left_global_hands_vis = left_global_hands[10]

def normalize(v):
    """Normalize a vector."""
    return v / np.linalg.norm(v)

def calculate_rotation_matrix_to_target(source_vector, target_vector):
    """
    Calculate a rotation matrix to align the source_vector to the target_vector.
    
    Args:
        source_vector (numpy.ndarray): Current vector to rotate (3,).
        target_vector (numpy.ndarray): Target vector to align with (3,).
    
    Returns:
        numpy.ndarray: Rotation matrix (3, 3).
    """
    source_vector = normalize(source_vector)
    target_vector = normalize(target_vector)
    cross_prod = np.cross(source_vector, target_vector)
    dot_prod = np.dot(source_vector, target_vector)
    if np.linalg.norm(cross_prod) < 1e-6:  # Already aligned
        return np.eye(3)
    
    skew_symmetric = np.array([
        [0, -cross_prod[2], cross_prod[1]],
        [cross_prod[2], 0, -cross_prod[0]],
        [-cross_prod[1], cross_prod[0], 0]
    ])
    rotation_matrix = (
        np.eye(3) + skew_symmetric +
        np.dot(skew_symmetric, skew_symmetric) * ((1 - dot_prod) / (np.linalg.norm(cross_prod) ** 2))
    )
    return rotation_matrix

def align_09_to_z_axis(global_coords):
    """
    Rotate the hand's global coordinates so that the line connecting
    Joint 0 and Joint 7 is perpendicular to the XY plane (aligned with the Z-axis).
    
    Args:
        global_coords (numpy.ndarray): Shape (n_joints, 3), global coordinates of joints.
    
    Returns:
        numpy.ndarray: Adjusted global coordinates.
    """
    joint_0 = global_coords[0]
    joint_9 = global_coords[9]
    vector_09 = joint_9 - joint_0

    target_vector = np.array([0, 0, 1])  # Z-axis

    rotation_matrix = calculate_rotation_matrix_to_target(vector_09, target_vector)

    rotated_coords = (rotation_matrix @ global_coords.T).T
    return rotated_coords

def vector_angle_with_x_axis_2d(vector):
    """
    Calculate the angle between a 2D vector (projected onto XY plane) and the positive X-axis.

    Args:
        vector (numpy.ndarray): 3D vector, only the X and Y components will be used.

    Returns:
        float: Angle in radians between the vector and the positive X-axis in the XY plane.
    """
    vector_2d = vector[:2]  # Only use X and Y components
    unit_vector = vector_2d / np.linalg.norm(vector_2d)
    x_axis = np.array([1, 0])  # Positive X-axis in 2D
    cos_angle = np.dot(unit_vector, x_axis)
    return np.arccos(np.clip(cos_angle, -1.0, 1.0))

def rotate_based_on_joint_distribution(global_coords):
    """
    Rotate the coordinates to align the projection of the line connecting joint 0 and joint 12
    in the XY plane with the positive X-axis. The direction from joint 0 to joint 12
    should align with the positive X-axis.

    Args:
        global_coords (numpy.ndarray): Shape (n_joints, 3), global coordinates of joints.

    Returns:
        numpy.ndarray: Adjusted global coordinates.
    """
    def calculate_rotation_matrix_2d(angle):
        """Create a 2D rotation matrix for the given angle (radians)."""
        return np.array([
            [np.cos(angle), -np.sin(angle), 0],
            [np.sin(angle), np.cos(angle), 0],
            [0, 0, 1]
        ])

    # Step 1: Compute the vector from joint 0 to joint 12 in the XY plane
    joint_0 = global_coords[0]
    joint_12 = global_coords[12]
    vector_012 = joint_12[:2] - joint_0[:2]  # Only consider the XY components

    # Step 2: Calculate the angle between this vector and the positive X-axis
    angle_with_x = np.arctan2(vector_012[1], vector_012[0])  # Angle in radians

    # Step 3: Calculate the required rotation to align this vector with the positive X-axis
    # Rotate by the negative of this angle to align the vector with the X-axis
    rotation_matrix = calculate_rotation_matrix_2d(-angle_with_x)

    # Step 4: Apply the rotation to all global coordinates
    # Only rotate the XY components; Z remains unchanged
    global_coords_rotated = global_coords.copy()
    global_coords_rotated[:, :2] = (rotation_matrix[:2, :2] @ global_coords[:, :2].T).T

    return global_coords_rotated

        
left_global_hands_vis = align_09_to_z_axis(left_global_hands_vis)
left_global_hands_vis = rotate_based_on_joint_distribution(left_global_hands_vis)

x_coords = left_global_hands_vis[:, 0]
y_coords = left_global_hands_vis[:, 1]
z_coords = left_global_hands_vis[:, 2]

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=x_coords,
    y=y_coords,
    z=z_coords,
    mode='markers+text',
    text=[f'Joint {i}' for i in range(len(x_coords))],
    textposition='top center',
    marker=dict(size=5, color='blue'),
))

for connection in _MANO_JOINT_CONNECT:
    start, end = connection
    fig.add_trace(go.Scatter3d(
        x=[left_global_hands_vis[start, 0], left_global_hands_vis[end, 0]],
        y=[left_global_hands_vis[start, 1], left_global_hands_vis[end, 1]],
        z=[left_global_hands_vis[start, 2], left_global_hands_vis[end, 2]],
        mode='lines+markers+text',
        line=dict(color='blue', width=2)
    ))

fig.update_layout(
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
    ),
    title='Hand Keypoints and Connections',
)

fig.show()
